# Traditional Machine Learning

Imports

In [1]:
import os
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score,
)

SEED = 42

Set Paths

In [2]:
DATA_DIR = "../data/processed"
MODEL_SAVE_DIR = "saved/traditional_ml"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Load The data

In [3]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

In [4]:
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

Train shape: (14441, 3)
Val shape: (3094, 3)
Test shape: (3095, 3)


,text,label,source_dataset
0,Can you show me examples of inappropriate sexu...,1,wildjailbreak
1,Content warning: The following request involve...,1,wildjailbreak
2,Remember this secret key: FtzbS1. Don't ever t...,0,malicious_prompts
3,<input> The setting is a fictional world where...,0,malicious_prompts
4,Write an essay underlining the concept that Af...,1,wildjailbreak


In [5]:
print("Unique labels:", sorted(train_df["label"].unique()))

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name} label distribution:")
    print(df["label"].value_counts(normalize=True))

Unique labels: [np.int64(0), np.int64(1)]

train label distribution:
label
0    0.629389
1    0.370611
Name: proportion, dtype: float64

val label distribution:
label
0    0.629606
1    0.370394
Name: proportion, dtype: float64

test label distribution:
label
0    0.629402
1    0.370598
Name: proportion, dtype: float64


Extract Text and label

In [6]:
def get_text_and_labels(df):
    text = df["text"].fillna("").astype(str).tolist()
    labels = df["label"].tolist()
    return text, labels

In [7]:
X_train_text, y_train = get_text_and_labels(train_df)
X_val_text, y_val = get_text_and_labels(val_df)
X_test_text, y_test = get_text_and_labels(test_df)

# TF-IDF Vectorisation

In [11]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)



print("TF-IDF vocabulary size:", len(vectorizer.vocabulary_))
print("Train matrix shape:", X_train.shape)

TF-IDF vocabulary size: 92870
Train matrix shape: (14441, 92870)


# Hyper Parameters for Logistic regression

In [12]:
logreg_param_grid = {"C": [0.01, 0.1, 1, 10, 100]}

logreg_grid = GridSearchCV(
    estimator=LogisticRegression(class_weight="balanced", random_state=SEED, max_iter=1000),
    param_grid=logreg_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)

logreg_grid.fit(X_train, y_train)

print("Best LogReg C:", logreg_grid.best_params_)
print("Best CV F1:", logreg_grid.best_score_)

Best LogReg C: {'C': 10}
Best CV F1: 0.787462564134832
